# AGENT EVAL SAMPLE

Load Agent run data from JSONL:

In [ ]:
from pathlib import Path
import sys
import time

candidates = [
    Path.cwd() / "solutions" / "src",
    Path.cwd().parent / "src",
    Path.cwd() / "src",
    Path.cwd().parent.parent / "solutions" / "src",
]
for path in candidates:
    if path.exists() and str(path) not in sys.path:
        sys.path.append(str(path))

from hackathon_solutions.config import load_config, validate_config
from hackathon_solutions.foundry_helpers import create_openai_client, load_jsonl

cfg = load_config()
validate_config(cfg)
openai_client = create_openai_client(cfg)
dataset_path = Path.cwd().parent / "data" / "session4_agent_data_tool_calls.jsonl"
if not dataset_path.exists():
    dataset_path = Path.cwd() / "solutions" / "data" / "session4_agent_data.jsonl"
rows = load_jsonl(dataset_path)
print("Rows loaded:", len(rows))
print("timestamp:", time.time())

Rows loaded: 1


Create an Agent run evaluation

In [20]:
# do some basic validation to ensure the dataset is in the expected format before creating the evaluation
required_fields = {"query", "response"}
missing = [idx for idx, row in enumerate(rows, start=1) if not required_fields.issubset(row.keys())]
if missing:
    raise ValueError(f"Rows missing required fields {required_fields}: {missing[:5]}")

schema_properties = {key: {"type": "string"} for key in sorted({k for row in rows for k in row.keys()})}
item_schema = {
    "type": "object",
    "properties": schema_properties,
    "required": ["query", "response"],
}

tool_definitions = [
    {
                "name": "fetch_weather",
                "description": "Fetches the weather information for the specified location.",
                "parameters": {
                    "type": "object",
                    "properties": {"location": {"type": "string", "description": "The location to fetch weather for."}},
                },
            },
]

evaluation = openai_client.evals.create(
    name=f"session4-eval-{int(time.time())}",
    data_source_config={
        "type": "custom",
        "item_schema": item_schema,
        "include_sample_schema": True,
    },
    # https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/azure-openai-graders
    testing_criteria=[
        {
        "type": "azure_ai_evaluator",
        "name": "task_adherence",
        "evaluator_name": "builtin.task_adherence",
        "initialization_parameters": {"deployment_name": cfg.model_deployment_name},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        }
            },
        {
        "type": "azure_ai_evaluator",
        "name": "tool_call_accuracy",
        "evaluator_name": "builtin.tool_call_accuracy",
        "initialization_parameters": {"deployment_name": f"{cfg.model_deployment_name}"},
        #"tool_definitions": tool_definitions,
        "data_mapping": {
            "query": "{{item.query}}",
            "tool_definitions": "{{item.tool_definitions}}",
            "tool_calls": "{{item.tool_calls}}",
            "response": "{{item.response}}",
        },
        }
    ]
)

evaluation.id

'eval_7403e39737b541d697133ba44044e0f2'

Run eval batches in the cloud

In [22]:
run = openai_client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"{evaluation.id}-run",
    data_source={
        "type": "responses",
        "model": cfg.model_deployment_name,
        "source": {
            "type": "file_content",
            "content": [{"item": row} for row in rows]
        },
        "input_messages": {
            "type": "template",
            "template": [{"role": "user", "content": "{{item.query}}"}]
        },
        "sampling_params": {"temperature": 0.0}
    }
)
while str(run.status).lower() in {"queued", "in_progress"}:
    time.sleep(8)
    run = openai_client.evals.runs.retrieve(run_id=run.id, eval_id=evaluation.id)

{
    "run_id": run.id,
    "status": run.status,
    "report_url": run.report_url,
    "result_counts": run.result_counts.model_dump() if run.result_counts else None
}

{'run_id': 'evalrun_d5d33d2f89834db397b2b836e4795d2a',
 'status': 'completed',
 'report_url': 'https://ai.azure.com/nextgen/r/TGmxZqOYSnWvA5cuV_bUiQ,SwedenAIML,,edwinfoundry0226,proj-default/build/evaluations/eval_7403e39737b541d697133ba44044e0f2/run/evalrun_d5d33d2f89834db397b2b836e4795d2a',
 'result_counts': {'errored': 1, 'failed': 0, 'passed': 0, 'total': 1}}